## tl;dr

**PARALLEL_DRY_RUN_RUNNING** with every locked absolute, matched-risk, data, signal-identity, and bias gate passing.

- V2R validation: 128 trades, 37.85%, PF 1.689, DD 9.35%.
- Matched V1R validation: 24.60%, PF 1.384, DD 8.13%.
- V2R return/DD improvement: 33.77%.
- V2R 2025: 7.52%, PF 1.342, DD 9.35%.
- V2R current 2026: 9.85%, PF 1.852, DD 4.61%.


## Context & Methods

This notebook validates the locked ATR/exposure risk overlay applied identically
to `DualTrendFuturesV1R` (control) and `DualTrendFuturesV2R` (challenger). It
recomputes raw metrics, verifies signal identity against the unscaled strategies,
applies absolute and matched-risk gates, checks bias evidence, and records local
dry-run state. All timestamps are UTC. Reused historical windows are not a new
sealed holdout.

### Key assumptions

- Research uses today's fixed six-pair universe and therefore has survivorship bias.
- Unlimited research stake differs from fixed-stake dry-run deployment.
- ATR controls stake only; it is not treated as a directional predictor.


## Data

### 1. Load the annotated matched-risk archives

In [1]:
from pathlib import Path
import base64
import json
import urllib.error
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
BACKTEST_DIR = PROJECT_ROOT / "user_data" / "backtest_results"
REPORT_DIR = PROJECT_ROOT / "user_data" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
STRATEGIES = ("DualTrendFuturesV1R", "DualTrendFuturesV2R")

records, raw = [], {}
for meta_path in sorted(BACKTEST_DIR.glob("*.meta.json")):
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    selected = [s for s in STRATEGIES if s in meta and meta[s].get("notes", "").startswith("phase=dual_v2r;")]
    if not selected:
        continue
    base = meta_path.name.removesuffix(".meta.json")
    zip_path = BACKTEST_DIR / f"{base}.zip"
    with zipfile.ZipFile(zip_path) as archive:
        result = json.loads(archive.read(f"{base}.json"))
    for strategy in selected:
        run_meta = meta[strategy]
        notes = run_meta["notes"]
        split = next(part.split("=", 1)[1] for part in notes.split(";") if part.startswith("split="))
        fee = float(next(part.split("=", 1)[1] for part in notes.split(";") if part.startswith("fee=")))
        stats = result["strategy"][strategy]
        records.append({
            "artifact": zip_path.name, "strategy": strategy, "split": split,
            "fee_per_side": fee, "run_epoch": int(run_meta["backtest_start_time"]),
            "trades": int(stats["total_trades"]), "long_trades": int(stats["trade_count_long"]),
            "short_trades": int(stats["trade_count_short"]),
            "return_pct": 100 * float(stats["profit_total"]),
            "profit_abs_usdt": float(stats["profit_total_abs"]),
            "profit_factor": float(stats["profit_factor"]),
            "max_drawdown_pct": 100 * float(stats["max_drawdown_account"]),
            "market_change_pct": 100 * float(stats["market_change"]),
        })
        raw[(strategy, split)] = stats

summary = pd.DataFrame(records).sort_values(["split", "strategy"]).reset_index(drop=True)
required_splits = {"train", "validation", "2024", "2025", "current_2026", "fee_train", "fee_validation"}
assert set(summary.strategy) == set(STRATEGIES)
assert all(set(summary.loc[summary.strategy == s, "split"]) == required_splits for s in STRATEGIES)
assert len(summary) == 14
summary.to_csv(REPORT_DIR / "dual_trend_v2r_backtest_summary.csv", index=False)
display(summary)

,artifact,strategy,split,fee_per_side,run_epoch,trades,long_trades,short_trades,return_pct,profit_abs_usdt,profit_factor,max_drawdown_pct,market_change_pct
0,backtest-result-2026-08-03_20-50-50.zip,DualTrendFuturesV1R,2024,0.0005,1785790247,71,50,21,21.157151,2115.715095,1.693323,8.048457,140.140040
1,backtest-result-2026-08-03_20-50-50.zip,DualTrendFuturesV2R,2024,0.0005,1785790248,65,43,22,28.261461,2826.146074,2.059263,8.739788,140.140040
2,backtest-result-2026-08-03_20-50-58.zip,DualTrendFuturesV1R,2025,0.0005,1785790255,68,42,26,2.763564,276.356431,1.099837,7.792406,-21.854841
3,backtest-result-2026-08-03_20-50-58.zip,DualTrendFuturesV2R,2025,0.0005,1785790257,63,37,26,7.517262,751.726229,1.342055,9.347932,-21.854841
4,backtest-result-2026-08-03_20-51-05.zip,DualTrendFuturesV1R,current_2026,0.0005,1785790263,38,0,38,7.333755,733.375534,1.495340,6.635442,-38.158032
5,backtest-result-2026-08-03_20-51-05.zip,DualTrendFuturesV2R,current_2026,0.0005,1785790264,34,0,34,9.847238,984.723849,1.851733,4.606070,-38.158032
6,backtest-result-2026-08-03_20-51-17.zip,DualTrendFuturesV1R,fee_train,0.0015,1785790271,203,97,106,127.707972,12770.797168,1.653597,19.920542,507.616606
7,backtest-result-2026-08-03_20-51-17.zip,DualTrendFuturesV2R,fee_train,0.0015,1785790274,192,92,100,99.785886,9978.588573,1.634399,18.784250,507.616606
8,backtest-result-2026-08-03_20-51-28.zip,DualTrendFuturesV1R,fee_validation,0.0015,1785790283,139,92,47,19.515823,1951.582312,1.297715,9.002344,84.531240
9,backtest-result-2026-08-03_20-51-28.zip,DualTrendFuturesV2R,fee_validation,0.0015,1785790285,128,80,48,32.829020,3282.901992,1.582454,9.958083,84.531240


### 2. Build the matched-risk comparison

In [2]:
control = summary[summary.strategy == "DualTrendFuturesV1R"].set_index("split")
challenger = summary[summary.strategy == "DualTrendFuturesV2R"].set_index("split")
comparison_rows = []
for split in ["train", "validation", "2024", "2025", "current_2026", "fee_train", "fee_validation"]:
    c, v = control.loc[split], challenger.loc[split]
    comparison_rows.append({
        "split": split, "v1r_trades": int(c.trades), "v2r_trades": int(v.trades),
        "v1r_return_pct": c.return_pct, "v2r_return_pct": v.return_pct,
        "return_delta_pp": v.return_pct - c.return_pct,
        "v1r_profit_factor": c.profit_factor, "v2r_profit_factor": v.profit_factor,
        "pf_delta": v.profit_factor - c.profit_factor,
        "v1r_max_drawdown_pct": c.max_drawdown_pct, "v2r_max_drawdown_pct": v.max_drawdown_pct,
        "drawdown_delta_pp": v.max_drawdown_pct - c.max_drawdown_pct,
        "v1r_return_to_dd": c.return_pct / c.max_drawdown_pct,
        "v2r_return_to_dd": v.return_pct / v.max_drawdown_pct,
        "return_to_dd_improvement_pct": 100 * ((v.return_pct / v.max_drawdown_pct) / (c.return_pct / c.max_drawdown_pct) - 1),
    })
comparison = pd.DataFrame(comparison_rows)
comparison.to_csv(REPORT_DIR / "dual_trend_v2r_vs_v1r.csv", index=False)
display(comparison)

,split,v1r_trades,v2r_trades,v1r_return_pct,v2r_return_pct,return_delta_pp,v1r_profit_factor,v2r_profit_factor,pf_delta,v1r_max_drawdown_pct,v2r_max_drawdown_pct,drawdown_delta_pp,v1r_return_to_dd,v2r_return_to_dd,return_to_dd_improvement_pct
0,train,203,192,141.072917,111.011174,-30.061743,1.734479,1.720009,-0.014470,18.194223,17.077879,-1.116344,7.753720,6.500291,-16.165527
1,validation,139,128,24.597154,37.846774,13.249620,1.383874,1.689225,0.305351,8.130667,9.352103,1.221436,3.025232,4.046873,33.770672
2,2024,71,65,21.157151,28.261461,7.104310,1.693323,2.059263,0.365940,8.048457,8.739788,0.691332,2.628721,3.233655,23.012464
3,2025,68,63,2.763564,7.517262,4.753698,1.099837,1.342055,0.242218,7.792406,9.347932,1.555527,0.354648,0.804163,126.749385
4,current_2026,38,34,7.333755,9.847238,2.513483,1.495340,1.851733,0.356392,6.635442,4.606070,-2.029372,1.105240,2.137883,93.431586
5,fee_train,203,192,127.707972,99.785886,-27.922086,1.653597,1.634399,-0.019199,19.920542,18.784250,-1.136292,6.410868,5.312210,-17.137430
6,fee_validation,139,128,19.515823,32.829020,13.313197,1.297715,1.582454,0.284739,9.002344,9.958083,0.955739,2.167860,3.296721,52.072585


### 3. Confirm market-data quality and risk constants

In [3]:
quality = pd.read_csv(REPORT_DIR / "dual_trend_data_quality.csv")
timeframe_reconciliation = pd.read_csv(REPORT_DIR / "dual_trend_timeframe_reconciliation.csv")
coverage_cols = [c for c in quality.columns if c.endswith("coverage_pct")]
assert quality[["duplicate_timestamps", "null_cells", "invalid_ohlcv"]].sum().sum() == 0
assert quality.largest_gap_candles.max() == 0
assert quality[coverage_cols].min().min() == 100.0
assert timeframe_reconciliation.mismatch_pct.max() <= 0.20

risk_test = json.loads((REPORT_DIR / "dual_trend_v2r_risk_callback_test.json").read_text(encoding="utf-8"))
source = (PROJECT_ROOT / "user_data" / "strategies" / "DualTrendFuturesRisk.py").read_text(encoding="utf-8")
assert risk_test["status"] == "PASS" and risk_test["exceptions_or_assertion_failures"] == 0
assert risk_test["minimum_multiplier"] == 0.325 and risk_test["maximum_multiplier"] == 0.65
for expected in (
    "risk_exposure_fraction = 0.65", "target_atr_pct = 0.03",
    "minimum_volatility_scalar = 0.50", "maximum_volatility_scalar = 1.00",
    "return proposed_stake * self.risk_exposure_fraction * volatility_scalar",
):
    assert expected in source

data_and_risk_check = {
    "same_audited_archives_as_v1_v2": True,
    "minimum_split_coverage_pct": float(quality[coverage_cols].min().min()),
    "duplicate_timestamps": int(quality.duplicate_timestamps.sum()),
    "null_cells": int(quality.null_cells.sum()),
    "invalid_ohlcv": int(quality.invalid_ohlcv.sum()),
    "largest_gap_candles": int(quality.largest_gap_candles.max()),
    "maximum_4h_1d_mismatch_pct": float(timeframe_reconciliation.mismatch_pct.max()),
    "risk_callback_test": risk_test["status"],
    "stake_multiplier_range": [risk_test["minimum_multiplier"], risk_test["maximum_multiplier"]],
}
(REPORT_DIR / "dual_trend_v2r_data_and_risk_check.json").write_text(json.dumps(data_and_risk_check, indent=2), encoding="utf-8")
data_and_risk_check

{'same_audited_archives_as_v1_v2': True,
 'minimum_split_coverage_pct': 100.0,
 'duplicate_timestamps': 0,
 'null_cells': 0,
 'invalid_ohlcv': 0,
 'largest_gap_candles': 0,
 'maximum_4h_1d_mismatch_pct': 0.1666666666666666,
 'risk_callback_test': 'PASS',
 'stake_multiplier_range': [0.325, 0.65]}

## Results

### 4. Recompute raw trade metrics and stake distributions

In [4]:
def recomputed_pf(values):
    gains = values[values > 0].sum()
    losses = -values[values < 0].sum()
    return float(gains / losses) if losses > 0 else float("inf")

reconciliation_rows, stake_rows = [], []
for (strategy, split), stats in raw.items():
    trades = pd.DataFrame(stats["trades"])
    reconciliation_rows.append({
        "strategy": strategy, "split": split,
        "stored_trades": int(stats["total_trades"]), "recomputed_trades": len(trades),
        "stored_profit_abs": float(stats["profit_total_abs"]), "recomputed_profit_abs": float(trades.profit_abs.sum()),
        "stored_pf": float(stats["profit_factor"]), "recomputed_pf": recomputed_pf(trades.profit_abs),
        "stored_wins": int(stats["wins"]), "recomputed_wins": int((trades.profit_abs > 0).sum()),
        "stored_losses": int(stats["losses"]), "recomputed_losses": int((trades.profit_abs < 0).sum()),
    })
    stake_rows.append({
        "strategy": strategy, "split": split, "trades": len(trades),
        "minimum_stake": float(trades.stake_amount.min()),
        "median_stake": float(trades.stake_amount.median()),
        "maximum_stake": float(trades.stake_amount.max()),
        "mean_stake": float(trades.stake_amount.mean()),
    })
reconciliation = pd.DataFrame(reconciliation_rows).sort_values(["strategy", "split"])
assert (reconciliation.stored_trades == reconciliation.recomputed_trades).all()
assert np.allclose(reconciliation.stored_profit_abs, reconciliation.recomputed_profit_abs, atol=1e-6)
assert np.allclose(reconciliation.stored_pf, reconciliation.recomputed_pf, atol=1e-12)
assert (reconciliation.stored_wins == reconciliation.recomputed_wins).all()
assert (reconciliation.stored_losses == reconciliation.recomputed_losses).all()
reconciliation.to_csv(REPORT_DIR / "dual_trend_v2r_backtest_reconciliation.csv", index=False)
stake_profile = pd.DataFrame(stake_rows).sort_values(["strategy", "split"])
stake_profile.to_csv(REPORT_DIR / "dual_trend_v2r_stake_profile.csv", index=False)
display(reconciliation)
display(stake_profile)

,strategy,split,stored_trades,recomputed_trades,stored_profit_abs,recomputed_profit_abs,stored_pf,recomputed_pf,stored_wins,recomputed_wins,stored_losses,recomputed_losses
4,DualTrendFuturesV1R,2024,71,71,2115.715095,2115.715095,1.693323,1.693323,25,25,46,46
6,DualTrendFuturesV1R,2025,68,68,276.356431,276.356431,1.099837,1.099837,26,26,42,42
8,DualTrendFuturesV1R,current_2026,38,38,733.375534,733.375534,1.495340,1.495340,13,13,25,25
10,DualTrendFuturesV1R,fee_train,203,203,12770.797168,12770.797168,1.653597,1.653597,68,68,135,135
12,DualTrendFuturesV1R,fee_validation,139,139,1951.582312,1951.582312,1.297715,1.297715,51,51,88,88
2,DualTrendFuturesV1R,train,203,203,14107.291663,14107.291663,1.734479,1.734479,71,71,132,132
0,DualTrendFuturesV1R,validation,139,139,2459.715429,2459.715429,1.383874,1.383874,51,51,88,88
5,DualTrendFuturesV2R,2024,65,65,2826.146074,2826.146074,2.059263,2.059263,27,27,38,38
7,DualTrendFuturesV2R,2025,63,63,751.726229,751.726229,1.342055,1.342055,29,29,34,34
9,DualTrendFuturesV2R,current_2026,34,34,984.723849,984.723849,1.851733,1.851733,13,13,21,21


,strategy,split,trades,minimum_stake,median_stake,maximum_stake,mean_stake
4,DualTrendFuturesV1R,2024,71,771.366750,1578.129840,1856.52564,1510.577159
6,DualTrendFuturesV1R,2025,68,1065.141900,1543.603260,1701.20167,1510.730312
8,DualTrendFuturesV1R,current_2026,38,1443.264300,1577.459730,1691.87161,1589.570746
10,DualTrendFuturesV1R,fee_train,203,759.604344,3383.128400,3996.39972,3071.863996
12,DualTrendFuturesV1R,fee_validation,139,769.642500,1649.734560,1984.35965,1633.319158
2,DualTrendFuturesV1R,train,203,760.990104,3470.970000,4120.07391,3162.775976
0,DualTrendFuturesV1R,validation,139,771.366750,1677.953700,2062.42274,1669.381073
5,DualTrendFuturesV2R,2024,65,817.608000,1558.564300,1949.16664,1500.175336
7,DualTrendFuturesV2R,2025,63,1037.111850,1517.183710,1740.95033,1514.074197
9,DualTrendFuturesV2R,current_2026,34,1443.264300,1596.173700,1725.28213,1601.562279


### 5. Verify that the overlay did not alter signals

In [5]:
def load_base_stats(strategy, phase, split):
    candidates = []
    for meta_path in BACKTEST_DIR.glob("*.meta.json"):
        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if strategy not in meta:
            continue
        notes = meta[strategy].get("notes", "")
        if not notes.startswith(f"phase={phase};") or f"split={split}" not in notes:
            continue
        base = meta_path.name.removesuffix(".meta.json")
        with zipfile.ZipFile(BACKTEST_DIR / f"{base}.zip") as archive:
            result = json.loads(archive.read(f"{base}.json"))
        candidates.append((int(meta[strategy]["backtest_start_time"]), result["strategy"][strategy]))
    assert candidates, (strategy, phase, split)
    return sorted(candidates)[-1][1]

identity_columns = ["pair", "open_date", "close_date", "is_short", "enter_tag", "exit_reason"]
signal_rows = []
for split in ["train", "validation", "2024", "2025", "current_2026", "fee_train", "fee_validation"]:
    for risk_strategy, base_strategy, phase in [
        ("DualTrendFuturesV1R", "DualTrendFuturesV1", "dual_v1"),
        ("DualTrendFuturesV2R", "DualTrendFuturesV2", "dual_v2"),
    ]:
        base_stats = load_base_stats(base_strategy, phase, split)
        risk_stats = raw[(risk_strategy, split)]
        base_ids = set(map(tuple, pd.DataFrame(base_stats["trades"])[identity_columns].astype(str).to_numpy()))
        risk_ids = set(map(tuple, pd.DataFrame(risk_stats["trades"])[identity_columns].astype(str).to_numpy()))
        signal_rows.append({
            "split": split, "risk_strategy": risk_strategy, "base_strategy": base_strategy,
            "base_trades": len(base_ids), "risk_trades": len(risk_ids),
            "missing_from_risk": len(base_ids - risk_ids), "extra_in_risk": len(risk_ids - base_ids),
            "identical": base_ids == risk_ids,
        })
signal_identity = pd.DataFrame(signal_rows)
assert signal_identity.identical.all()
signal_identity.to_csv(REPORT_DIR / "dual_trend_v2r_signal_identity.csv", index=False)
display(signal_identity)

,split,risk_strategy,base_strategy,base_trades,risk_trades,missing_from_risk,extra_in_risk,identical
0,train,DualTrendFuturesV1R,DualTrendFuturesV1,203,203,0,0,True
1,train,DualTrendFuturesV2R,DualTrendFuturesV2,192,192,0,0,True
2,validation,DualTrendFuturesV1R,DualTrendFuturesV1,139,139,0,0,True
3,validation,DualTrendFuturesV2R,DualTrendFuturesV2,128,128,0,0,True
4,2024,DualTrendFuturesV1R,DualTrendFuturesV1,71,71,0,0,True
5,2024,DualTrendFuturesV2R,DualTrendFuturesV2,65,65,0,0,True
6,2025,DualTrendFuturesV1R,DualTrendFuturesV1,68,68,0,0,True
7,2025,DualTrendFuturesV2R,DualTrendFuturesV2,63,63,0,0,True
8,current_2026,DualTrendFuturesV1R,DualTrendFuturesV1,38,38,0,0,True
9,current_2026,DualTrendFuturesV2R,DualTrendFuturesV2,34,34,0,0,True


### 6. Apply all absolute and matched-risk gates

In [6]:
v = challenger
c = control
val_stats = raw[("DualTrendFuturesV2R", "validation")]
pair_rows = pd.DataFrame([r for r in val_stats["results_per_pair"] if r["key"] != "TOTAL"])
val_trades = pd.DataFrame(val_stats["trades"])
positive_pairs = pair_rows[pair_rows.profit_total_abs > 0]
pair_concentration = float(positive_pairs.profit_total_abs.max() / positive_pairs.profit_total_abs.sum())
positive_trades = val_trades[val_trades.profit_abs > 0]
trade_concentration = float(positive_trades.profit_abs.max() / positive_trades.profit_abs.sum())

gates = {
    "train_positive": v.loc["train", "return_pct"] > 0,
    "train_pf_1_20": v.loc["train", "profit_factor"] >= 1.20,
    "train_dd_30": v.loc["train", "max_drawdown_pct"] <= 30,
    "validation_100_trades": v.loc["validation", "trades"] >= 100,
    "validation_positive": v.loc["validation", "return_pct"] > 0,
    "validation_pf_1_50": v.loc["validation", "profit_factor"] >= 1.50,
    "validation_dd_12": v.loc["validation", "max_drawdown_pct"] <= 12,
    "year_2024": (v.loc["2024", "return_pct"] > 0 and v.loc["2024", "profit_factor"] >= 1.20 and v.loc["2024", "max_drawdown_pct"] <= 12 and v.loc["2024", "trades"] >= 20),
    "year_2025": (v.loc["2025", "return_pct"] > 0 and v.loc["2025", "profit_factor"] >= 1.15 and v.loc["2025", "max_drawdown_pct"] <= 12 and v.loc["2025", "trades"] >= 20),
    "current_2026": (v.loc["current_2026", "return_pct"] > 0 and v.loc["current_2026", "profit_factor"] >= 1.30 and v.loc["current_2026", "max_drawdown_pct"] <= 10 and v.loc["current_2026", "trades"] >= 12),
    "fee_validation": (v.loc["fee_validation", "return_pct"] > 0 and v.loc["fee_validation", "profit_factor"] >= 1.30 and v.loc["fee_validation", "max_drawdown_pct"] <= 13),
    "fee_train": v.loc["fee_train", "return_pct"] > 0 and v.loc["fee_train", "profit_factor"] >= 1.20,
    "validation_five_positive_pairs": int((pair_rows.profit_total > 0).sum()) >= 5,
    "validation_pair_concentration": pair_concentration <= 0.50,
    "validation_trade_concentration": trade_concentration <= 0.25,
    "matched_validation_return": v.loc["validation", "return_pct"] > c.loc["validation", "return_pct"],
    "matched_validation_pf": v.loc["validation", "profit_factor"] > c.loc["validation", "profit_factor"],
    "matched_validation_return_dd_20pct": (v.loc["validation", "return_pct"] / v.loc["validation", "max_drawdown_pct"]) >= 1.20 * (c.loc["validation", "return_pct"] / c.loc["validation", "max_drawdown_pct"]),
    "matched_2025_return": v.loc["2025", "return_pct"] > c.loc["2025", "return_pct"],
    "matched_current_return": v.loc["current_2026", "return_pct"] > c.loc["current_2026", "return_pct"],
    "matched_fee_validation_return": v.loc["fee_validation", "return_pct"] > c.loc["fee_validation", "return_pct"],
    "matched_fee_validation_pf": v.loc["fee_validation", "profit_factor"] > c.loc["fee_validation", "profit_factor"],
}
gates_df = pd.DataFrame([{"gate": key, "pass": bool(value)} for key, value in gates.items()])
assert gates_df["pass"].all()
gates_df.to_csv(REPORT_DIR / "dual_trend_v2r_performance_gates.csv", index=False)
pair_audit = pair_rows[["key", "trades", "profit_total_pct", "profit_factor"]].copy()
pair_audit.to_csv(REPORT_DIR / "dual_trend_v2r_validation_pairs.csv", index=False)
concentration = {"positive_pairs": int((pair_rows.profit_total > 0).sum()), "pair_profit_concentration": pair_concentration, "winning_trade_concentration": trade_concentration}
(REPORT_DIR / "dual_trend_v2r_concentration.json").write_text(json.dumps(concentration, indent=2), encoding="utf-8")
display(pair_audit)
display(gates_df)
concentration

,key,trades,profit_total_pct,profit_factor
0,XRP/USDT:USDT,20,12.10,2.449587
1,BTC/USDT:USDT,20,7.05,2.836475
2,DOGE/USDT:USDT,23,6.84,1.462581
3,BNB/USDT:USDT,20,4.58,1.566887
4,ETH/USDT:USDT,22,4.35,1.421972
5,ADA/USDT:USDT,23,2.93,1.306970


,gate,pass
0,train_positive,True
1,train_pf_1_20,True
2,train_dd_30,True
3,validation_100_trades,True
4,validation_positive,True
5,validation_pf_1_50,True
6,validation_dd_12,True
7,year_2024,True
8,year_2025,True
9,current_2026,True


{'positive_pairs': 6,
 'pair_profit_concentration': 0.3195947681380584,
 'winning_trade_concentration': 0.13250466218687443}

### 7. Validate bias checks and inspect dry-run runtime

In [7]:
lookahead = pd.read_csv(REPORT_DIR / "DualTrendFuturesV2R_lookahead.csv").iloc[0]
technical = json.loads((REPORT_DIR / "dual_trend_v2r_technical_checks.json").read_text(encoding="utf-8"))
technical_pass = (
    str(lookahead.has_bias).lower() == "false" and int(lookahead.total_signals) == 100
    and int(lookahead.biased_entry_signals) == 0 and int(lookahead.biased_exit_signals) == 0
    and technical["recursive_analysis"]["status"] == "PASS"
    and not technical["recursive_analysis"]["indicator_lookahead_found"]
    and abs(float(technical["recursive_analysis"]["ema200_1d_variance_pct_at_800"])) <= 0.01
)
assert technical_pass

base_config = json.loads((PROJECT_ROOT / "user_data" / "config.json").read_text(encoding="utf-8"))
assert base_config["exchange"]["key"] == base_config["exchange"]["secret"] == ""
credentials = f'{base_config["api_server"]["username"]}:{base_config["api_server"]["password"]}'

def runtime_at(port):
    request = urllib.request.Request(f"http://127.0.0.1:{port}/api/v1/show_config")
    request.add_header("Authorization", "Basic " + base64.b64encode(credentials.encode()).decode())
    try:
        with urllib.request.urlopen(request, timeout=5) as response:
            cfg = json.loads(response.read())
        return {"reachable": True, "http_status": int(response.status), "state": cfg["state"],
                "dry_run": bool(cfg["dry_run"]), "strategy": cfg["strategy"],
                "trading_mode": cfg["trading_mode"], "short_allowed": bool(cfg["short_allowed"]),
                "stake_amount": float(cfg["stake_amount"]), "max_open_trades": int(cfg["max_open_trades"])}
    except (urllib.error.URLError, TimeoutError):
        return {"reachable": False}

runtime_v1 = runtime_at(8080)
runtime_v2r = runtime_at(8081)
assert runtime_v1["reachable"] and runtime_v1["state"] == "running" and runtime_v1["dry_run"] and runtime_v1["strategy"] == "DualTrendFuturesV1"
runtime_v2r_ok = (runtime_v2r.get("reachable", False) and runtime_v2r.get("state") == "running"
                  and runtime_v2r.get("dry_run") and runtime_v2r.get("strategy") == "DualTrendFuturesV2R"
                  and runtime_v2r.get("trading_mode") == "futures" and runtime_v2r.get("short_allowed")
                  and runtime_v2r.get("stake_amount") == 25 and runtime_v2r.get("max_open_trades") == 2)
runtime_evidence = {"v1_port_8080": runtime_v1, "v2r_port_8081": runtime_v2r, "v2r_runtime_verified": bool(runtime_v2r_ok)}
(REPORT_DIR / "dual_trend_v2r_runtime_verification.json").write_text(json.dumps(runtime_evidence, indent=2), encoding="utf-8")
runtime_evidence

{'v1_port_8080': {'reachable': True,
  'http_status': 200,
  'state': 'running',
  'dry_run': True,
  'strategy': 'DualTrendFuturesV1',
  'trading_mode': 'futures',
  'short_allowed': True,
  'stake_amount': 25.0,
  'max_open_trades': 2},
 'v2r_port_8081': {'reachable': True,
  'http_status': 200,
  'state': 'running',
  'dry_run': True,
  'strategy': 'DualTrendFuturesV2R',
  'trading_mode': 'futures',
  'short_allowed': True,
  'stake_amount': 25.0,
  'max_open_trades': 2},
 'v2r_runtime_verified': True}

## Takeaways

- The identical 32.5%-65% stake overlay reduced validation drawdown below 10% for both control and challenger.
- V2R retained superior validation return, PF, return/DD, weak-2025 return, current-2026 return, and fee-stress performance versus V1R.
- Trade identities are exactly unchanged from V1/V2 across all seven runs; only stake sizing changed.
- Raw rows reconcile exactly, the reused market-data audit remains clean, and 100-signal lookahead plus recursive checks pass.
- Historical selection evidence is not a fresh holdout. Any parallel dry-run is forward evidence collection, not proof of future profit.


In [8]:
all_gates_pass = bool(gates_df["pass"].all() and technical_pass and signal_identity.identical.all())
decision_name = "PARALLEL_DRY_RUN_RUNNING" if all_gates_pass and runtime_v2r_ok else "PARALLEL_DRY_RUN_APPROVED_NOT_STARTED"
decision = {
    "decision": decision_name, "strategy": "DualTrendFuturesV2R", "control": "DualTrendFuturesV1R",
    "active_primary_strategy": runtime_v1["strategy"], "all_performance_gates_pass": bool(gates_df["pass"].all()),
    "technical_gates_pass": bool(technical_pass), "signal_identity_pass": bool(signal_identity.identical.all()),
    "risk_callback_test_pass": risk_test["status"] == "PASS", "parallel_runtime_verified": bool(runtime_v2r_ok),
    "live_trading_authorized": False, "live_api_keys_present": False, "new_sealed_holdout": False,
    "future_profit_guaranteed": False,
}
assert all_gates_pass
(REPORT_DIR / "dual_trend_v2r_validation_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
decision

{'decision': 'PARALLEL_DRY_RUN_RUNNING',
 'strategy': 'DualTrendFuturesV2R',
 'control': 'DualTrendFuturesV1R',
 'active_primary_strategy': 'DualTrendFuturesV1',
 'all_performance_gates_pass': True,
 'technical_gates_pass': True,
 'signal_identity_pass': True,
 'risk_callback_test_pass': True,
 'parallel_runtime_verified': True,
 'live_trading_authorized': False,
 'live_api_keys_present': False,
 'new_sealed_holdout': False,
 'future_profit_guaranteed': False}